In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT = Path("/content/uber-ride-analytics")
PROCESSED = PROJECT / "data" / "processed"
VISUALS = PROJECT / "visuals"

VISUALS.mkdir(parents=True, exist_ok=True)

clean_df = pd.read_csv(
    PROCESSED / "uber_rides_cleaned.csv"
)

sns.set_theme(style="whitegrid")

def save_chart(filename):
    plt.tight_layout()
    plt.savefig(
        VISUALS / filename,
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

In [ ]:

if "booking_status" in clean_df.columns:
    cancellation_df = clean_df[
        clean_df["booking_status"].astype("string")
        .str.contains("cancel", case=False, na=False)
    ]

    print("Cancelled booking records:", len(cancellation_df))

    if "booking_hour" in cancellation_df.columns:
        cancelled_hourly = (
            cancellation_df["booking_hour"]
            .dropna().astype(int)
            .value_counts()
            .reindex(range(24), fill_value=0)
        )

        plt.figure(figsize=(12, 5))
        plt.bar(cancelled_hourly.index, cancelled_hourly.values)
        plt.xticks(range(24))
        plt.title("Cancelled Bookings by Hour")
        plt.xlabel("Hour of Day")
        plt.ylabel("Cancelled Bookings")
        save_chart("cancellations_by_hour.png")

    # Inspect cancellation reason columns
    reason_columns = [
        col for col in clean_df.columns
        if "reason" in col
    ]

    for col in reason_columns:
        print(f"\nTop values in {col}:")
        display(cancellation_df[col].value_counts().head(10))


In [ ]:

excel_path = PROJECT / "uber_ride_analytics.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    clean_df.to_excel(
        writer, sheet_name="Cleaned Data", index=False
    )

    pd.DataFrame(
        list(kpis.items()), columns=["KPI", "Value"]
    ).to_excel(
        writer, sheet_name="KPI Summary", index=False
    )

    if "booking_status" in clean_df.columns:
        clean_df["booking_status"].value_counts().rename_axis(
            "Booking Status"
        ).reset_index(name="Bookings").to_excel(
            writer, sheet_name="Booking Status", index=False
        )

    if "payment_method" in clean_df.columns:
        clean_df["payment_method"].value_counts().rename_axis(
            "Payment Method"
        ).reset_index(name="Bookings").to_excel(
            writer, sheet_name="Payment Methods", index=False
        )

print("Excel file created:", excel_path)
